In [ ]:
%load_ext autoreload
%autoreload 2

In [14]:
import mlflow
mlflow.set_experiment("MLflow Quickstart")
mlflow.set_tracking_uri("http://127.0.0.1:5000") 

In [25]:

import pandas as pd
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from utils.utils import wrapper_logging, load_model_by_alias,load_model_by_version
from utils.dataprocessing import get_pipeline, prepare_features

In [ ]:
def get_columns():
    return datasets.load_iris().feature_names

def load_dataset():
    # Load the Iris dataset
    X, y = datasets.load_iris(return_X_y=True)
    # Split the data into training and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    return X_train,y_train, X_test,y_test

def get_hyperparameters():
    return {
        "solver": "lbfgs",
        "max_iter": 1000,
        "random_state": 8888,
    }

def init_model(params):
    # Enable autologging for scikit-learn
    # mlflow.sklearn.autolog()

    # Just train the model normally
    lr = LogisticRegression(**params)
    return lr

def predict(model, X, y):
    y_pred = model.predict(X)
    return model, (y, y_pred)

def load_model(file_path):
    loaded_model = mlflow.pyfunc.load_model(file_path)
    return loaded_model

def visual_df(X_test, y_test, model, columns):
    y_pred = model.predict(X_test)
    df = pd.DataFrame(X_test, columns=columns)
    df["pred"] = y_pred
    df["actual"] = y_test
    print(df[:4])

# can use decorator in here
def training_loop(X_train,y_train,X_test,y_test, params):
    model = init_model(params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return model, y_pred, y_test

def test_one_example(model, X_test, y_test):
    idx = 0
    y_pred = model.predict(X_test[idx].reshape(1, -1))
    print("Predict", X_test[idx], y_pred[-1], y_test[idx])

### Ingest Datasets

In [28]:
X_train,y_train, X_test,y_test = load_dataset()

params = get_hyperparameters()
columns = get_columns()
print(columns)


['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']


### Validation

In [30]:
FEATURES = [
    "sepal length (cm)",
    "sepal width (cm)",
    "petal length (cm)",
    "petal width (cm)",
]
train_df = pd.DataFrame(X_train, columns=columns)
test_df = pd.DataFrame(X_test, columns=columns)
X_train = prepare_features(train_df, FEATURES)
X_test = prepare_features(test_df, FEATURES)

### Preprocessing & training

In [31]:
model = load_model_by_alias("production")
pl = get_pipeline(train_df, model)
pl.fit(X_train,y_train)
pred = pl.predict(X_test)
print(pred[0], y_test[0])

model_uri models:/iris_model@production
1 1


In [ ]:
y_pred = model.predict(X_test.reshape(1, -1))
print("Predict", X_test, y_pred[-1], y_test)